# Training: Between-Participant Variability

**Scope:**

- **Subjects:** VALID_SUBJECTS, [1, 2, 4, 6, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22]
- **Channel:** Fz
- **Time window:** 2 s  (following prior segment-length optimization)
- **Loss:** MSE (fixed, consistent with previous experiments)
- **CV:** GroupKFold, 5 folds over 40 trials
- **Train overlap:** 50% | **Test overlap:** 0%

**Settings:**

- 10 epochs, batch size 16
- Adam: lr = 1e-4, weight_decay = 1e-5
- LR decayed by 0.1 every 5 epochs
- Identical intra-subject training protocol for all subjects

**Purpose:**

Characterizes subject-specific performance variability under identical
intra-subject conditions. Population-level generalization is left to
future work with larger and more diverse samples.

**Runs:**

- One model per subject
- Exp starts: S2=196, S4=191, S6=171, S9=166, S12=186, S17=176, S21=181

**Output directory:** `Between-Participant-Variability/Results/`

**Model directory:** `Between-Participant-Variability/Models/`

In [ ]:
# ============================================================
# 0. Environment setup (Colab or local)
# ============================================================
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    try:
        get_ipython().run_line_magic('matplotlib', 'inline')
    except Exception:
        pass
    !pip install -q pims


# ============================================================
# 1. Standard library
# ============================================================
import os
import sys
import pickle
import traceback

# ============================================================
# 2. Third-party
# ============================================================
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# 3. Project paths
# ============================================================
if IN_COLAB:
    SOURCE_ROOT = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Source'
    DATA_ROOT   = '/content/drive/MyDrive/Colab Notebooks/Face2EEG/Data'
else:
    SOURCE_ROOT = r'D:\Thesis\Codes\Source'
    DATA_ROOT   = r'D:\Thesis\Codes\Data'

# sanity check
if not os.path.isdir(os.path.join(SOURCE_ROOT, 'face2eeg')):
    raise FileNotFoundError(
        f"face2eeg package not found under SOURCE_ROOT = {SOURCE_ROOT}. "
        f"Expected directory: {os.path.join(SOURCE_ROOT, 'face2eeg')}"
    )

if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)


# ============================================================
# 4. Package imports
# ============================================================
from face2eeg.core.experiments import run_kfold_pipeline
from face2eeg.core.models      import try_gpu
from face2eeg.core.data        import clear_video_cache, clear_signals_cache

# Framework constants (from config)
from face2eeg.core.config import (
    SEED,
    FS_EEG,
    FS_VIDEO,
    EXCLUDED_SUBJECTS,
    VALID_SUBJECTS
)


# ============================================================
# 5. Reproducibility
# ============================================================
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ============================================================
# Participant Selection
# ============================================================
# SUBJECT_POOL           = [s for s in range(1, 23) if s not in EXCLUDED_SUBJECTS]
# N_SELECTED_SUBJECTS    = 7
# SUBJECT_SELECTION_SEED = 5792

# _rng_subjects = np.random.RandomState(SUBJECT_SELECTION_SEED)
# SUBJECTS = sorted(_rng_subjects.choice(SUBJECT_POOL, size=N_SELECTED_SUBJECTS, replace=False).tolist())
# del _rng_subjects

# print("SUBJECTS    :", SUBJECTS)

In [ ]:
# ============================================================
# Configuration
# ============================================================
BASE = os.path.join(DATA_ROOT, 'Experiments', 'Cross-Validation')

EXP_DIR     = os.path.join(BASE, 'Between-Participant-Variability')
RESULTS_DIR = os.path.join(EXP_DIR, 'Results')
MODELS_DIR  = os.path.join(EXP_DIR, 'Models')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

# ============================================================
# Experiment scope
# ============================================================
SUBJECTS = VALID_SUBJECTS
CHANNEL  = 'Fz'

# ============================================================
# Hyperparameters
# ============================================================
TIME_WINDOW  = 2
FRAME_SIZE   = (32, 32)

EPOCHS       = 10
BATCH_SIZE   = 16
LR           = 1e-4
WEIGHT_DECAY = 1e-5
SCHED_STEP   = 5
SCHED_GAMMA  = 0.1

# Data generation 
TRAIN_OVERLAP       = True
TRAIN_OVERLAP_RATIO = 0.5
TEST_OVERLAP        = False
TEST_OVERLAP_RATIO  = 0.0

# Unique exp_start per subject (avoids checkpoint collisions)
EXP_START_BASE = 200

# EXP_START_PER_SUBJECT = {
#     2:  196,
#     4:  191,
#     6:  171,
#     9:  166,
#     12: 186,
#     17: 176,
#     21: 181,
# }

# REMAINING_SUBJECTS = list(SUBJECTS - EXP_START_PER_SUBJECT.keys())

# EXP_START_BASE = 200
# EXP_START_PER_SUBJECT |= {
#     sid: EXP_START_BASE + i * 10 for i, sid in enumerate(REMAINING_SUBJECTS)
# }

EXP_START_PER_SUBJECT = {
    sid: EXP_START_BASE + i * 10 for i, sid in enumerate(SUBJECTS)
}

# ============================================================
# Device
# ============================================================
device = try_gpu()


# ============================================================
# Sanity check
# ============================================================
print("EXP_DIR     :", EXP_DIR)
print("RESULTS_DIR :", RESULTS_DIR)
print("MODELS_DIR  :", MODELS_DIR)
print()
print("CHANNEL     :", CHANNEL)
print("TIME_WINDOW :", TIME_WINDOW)
print("FRAME_SIZE  :", FRAME_SIZE)
print()
print("EPOCHS      :", EPOCHS)
print("BATCH_SIZE  :", BATCH_SIZE)
print("LR          :", LR)
print("WEIGHT_DECAY:", WEIGHT_DECAY)
print("SCHED_STEP  :", SCHED_STEP)
print("SCHED_GAMMA :", SCHED_GAMMA)
print()
print("TRAIN_OVERLAP       :", TRAIN_OVERLAP)
print("TRAIN_OVERLAP_RATIO :", TRAIN_OVERLAP_RATIO)
print("TEST_OVERLAP        :", TEST_OVERLAP)
print()
print("EXP_START_PER_SUBJECT:")
for sid in SUBJECTS:
    print(f"  S{sid:02d}  →  exp_start = {EXP_START_PER_SUBJECT[sid]}")
print()
print("FS_EEG      :", FS_EEG, " (from config)")
print("FS_VIDEO    :", FS_VIDEO, " (from config)")
print("device      :", device)

## Run: one model per subject

Loops over the 7 subjects. Each run:
- Uses the same 5-fold GroupKFold splits
- Trains a fresh PhysNet-3DCNN-ED with MSE loss
- Saves results under `Results/sub{XX}_chFz.pkl`
- Skips automatically if the file already exists
- Frees RAM between subjects

In [ ]:
results_all = {}   # {subject_id: results_dict}

for subject_id in SUBJECTS:
    print(f"\n{'#'*70}")
    print(f"#  SUBJECT {subject_id:02d}/{SUBJECTS[-1]:02d}")
    print(f"{'#'*70}")

    # Free RAM between subjects
    clear_video_cache()
    clear_signals_cache()

    # ---------- skip if already done ----------
    result_path = os.path.join(
        RESULTS_DIR, f'sub{subject_id:02d}_ch{CHANNEL}.pkl'
    )
    if os.path.isfile(result_path):
        print(f"  [skip] already exists: {result_path}")
        continue

    print(f"  exp_start = {EXP_START_PER_SUBJECT[subject_id]}")

    # ---------- run CV ----------
    try:
        results = run_kfold_pipeline(
            subject_id=subject_id,
            channel_name=CHANNEL,
            time_window=TIME_WINDOW,

            # CV splits
            n_points=40, n_splits=5,

            # scaler
            scaler_method='z-score',

            # data generation
            frame_size=FRAME_SIZE,
            video_fr=FS_VIDEO,
            signal_fs=FS_EEG,
            y_type='RawEEG',

            # train (50% overlap)
            train_overlap=TRAIN_OVERLAP,
            train_overlap_ratio=TRAIN_OVERLAP_RATIO,
            train_shuffle_output=True,
            train_signal_normalize=True,

            # test (no overlap)
            test_overlap=TEST_OVERLAP,
            test_overlap_ratio=TEST_OVERLAP_RATIO,
            test_shuffle_output=False,
            test_signal_normalize=True,

            # model
            video_length=None,
            signal_length=None,

            # training
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            learning_rate=LR,
            weight_decay=WEIGHT_DECAY,
            loss_fn=nn.MSELoss(),
            scheduler_step_size=SCHED_STEP,
            scheduler_gamma=SCHED_GAMMA,

            # prediction
            predict_batch_size=BATCH_SIZE,

            # output
            model_exp_start=EXP_START_PER_SUBJECT[subject_id],
            save_dir=MODELS_DIR,
            device=device,
            return_history=True,
            verbose=True,
            root_dir=DATA_ROOT,
        )

    except Exception as e:
        print(f"  !! ERROR for subject={subject_id}: {e}")
        traceback.print_exc()
        continue   # next subject

    # ---------- store ----------
    results_all[subject_id] = results

    # ---------- save ----------
    with open(result_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"  [saved] {result_path}")

    # ---------- free GPU memory ----------
    if device.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
print("\n" + "="*70)
print("DONE — Between-Participant Variability")
print("="*70)
print(f"Results: {RESULTS_DIR}")
print(f"Models : {MODELS_DIR}")
print()

for sid in SUBJECTS:
    p = os.path.join(RESULTS_DIR, f'sub{sid:02d}_ch{CHANNEL}.pkl')
    ok = "✅" if os.path.isfile(p) else "❌"
    print(f"  {ok}  S{sid:02d}  {os.path.basename(p)}")